# CineSense Machine Learning Project\nThis notebook uses **Genre, Style, Age Restriction, and Era**. No user star ratings are used.

## Important\nThe movie catalog can support content-based recommendations immediately. Accuracy, Precision, Recall, F1-Score, and Confusion Matrices for predicting what a person actually chooses require genuine preference-selection labels.

In [ ]:
import pandas as pd\nimport matplotlib.pyplot as plt\nfrom sklearn.compose import ColumnTransformer\nfrom sklearn.preprocessing import OneHotEncoder\nfrom sklearn.pipeline import Pipeline\nfrom sklearn.model_selection import train_test_split\nfrom sklearn.neighbors import KNeighborsClassifier\nfrom sklearn.tree import DecisionTreeClassifier\nfrom sklearn.ensemble import RandomForestClassifier\nfrom sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay\n

## 1. Load the cleaned CineSense movie dataset

In [ ]:
movies = pd.read_excel('CineSense_Cleaned_Dataset.xlsx', sheet_name='Movies')\nmovies.head()

## 2. Data Cleaning Checks

In [ ]:
movies = movies.drop_duplicates(subset=['Movie_ID']).dropna(subset=['Title','Genre','Style','Age_Restriction','Release_Year','Era']).copy()\nmovies.info()

## 3. Exploratory Data Analysis (EDA)

In [ ]:
print(movies['Style'].value_counts())\nprint(movies['Age_Restriction'].value_counts())\nprint(movies['Era'].value_counts())\nmovies['Genre'].str.split('|').explode().value_counts()

In [ ]:
movies['Era'].value_counts().plot(kind='bar', title='Movies by Era')\nplt.ylabel('Number of Movies')\nplt.show()

## 4. Content-Based Recommendation\nEach matched criterion contributes 25%, so the score is transparent rather than a fabricated ML probability.

In [ ]:
def recommend_movies(genre, style, age, era, top_n=5):\n    result = movies.copy()\n    result['Genre_Match'] = result['Genre'].str.split('|').apply(lambda x: genre in x).astype(int)\n    result['Style_Match'] = (result['Style'] == style).astype(int)\n    result['Age_Match'] = (result['Age_Restriction'] == age).astype(int)\n    result['Era_Match'] = (result['Era'] == era).astype(int)\n    result['Match_Percent'] = 25 * result[['Genre_Match','Style_Match','Age_Match','Era_Match']].sum(axis=1)\n    return result.sort_values(['Match_Percent','Title'], ascending=[False,True])[['Title','Genre','Style','Age_Restriction','Era','Match_Percent']].head(top_n)\nrecommend_movies('Drama','Live-Action','R','1990s')

## 5. KNN, Decision Tree, and Random Forest\nFill the preference workbook with genuine selections before running this section.

In [ ]:
prefs = pd.read_excel('CineSense_Preference_Collection.xlsx', sheet_name='Preference Collection')\nneeded = ['Preferred_Genre','Preferred_Style','Preferred_Age_Restriction','Preferred_Era','Selected_Movie_ID']\nprefs = prefs.dropna(subset=needed).copy()\nprint('Usable labeled selections:', len(prefs))

In [ ]:
if len(prefs) < 20 or prefs['Selected_Movie_ID'].nunique() < 2:\n    print('Collect more genuine preference selections before supervised evaluation.')\nelse:\n    X = prefs[['Preferred_Genre','Preferred_Style','Preferred_Age_Restriction','Preferred_Era']]\n    y = prefs['Selected_Movie_ID'].astype(str)\n    stratify = y if y.value_counts().min() >= 2 else None\n    X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.20,random_state=42,stratify=stratify)\n    pre = ColumnTransformer([('cat', OneHotEncoder(handle_unknown='ignore'), list(X.columns))])\n    models = {'KNN':KNeighborsClassifier(n_neighbors=max(1,min(5,len(X_train)))), 'Decision Tree':DecisionTreeClassifier(random_state=42), 'Random Forest':RandomForestClassifier(n_estimators=200,random_state=42)}\n    results=[]\n    for name, model in models.items():\n        pipe=Pipeline([('preprocess',pre),('model',model)])\n        pipe.fit(X_train,y_train)\n        pred=pipe.predict(X_test)\n        results.append([name,accuracy_score(y_test,pred),precision_score(y_test,pred,average='macro',zero_division=0),recall_score(y_test,pred,average='macro',zero_division=0),f1_score(y_test,pred,average='macro',zero_division=0)])\n        labels=sorted(y.unique())\n        ConfusionMatrixDisplay(confusion_matrix(y_test,pred,labels=labels),display_labels=labels).plot(xticks_rotation=90)\n        plt.title(name + ' Confusion Matrix')\n        plt.show()\n    comparison=pd.DataFrame(results,columns=['Model','Accuracy','Precision','Recall','F1-Score'])\n    display(comparison)